In [ ]:
from unsloth import FastVisionModel
from dotenv import load_dotenv
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from jiwer import wer, cer
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image, ImageEnhance

In [ ]:
load_dotenv()

In [ ]:
# TODO: update with front/back details
field_structure = {
    "first_name": "string (arabic)",
    "last_name": "string (arabic)",
    "national_id": "string, 14 digits",
    "address": "string (arabic)",
    "address2": "string (arabic)",
    "birthdate": "string, formatted date (arabic)",
    "issue_date": "string, formatted date (arabic)",
    "expiration_date": "string, formatted date (arabic)",
    "job_title": "string (arabic)",
    "gender": "string, 'male' or 'female' (arabic)",
    "religion": "string, 'muslim' or 'christian' (arabic)",
    "marital_status": "string, 'single', 'married' or 'widow' (arabic)"
}

In [ ]:
SYSTEM_PROMPT = f'''
    You are a Vision Language Model tasked with extracted field values from an Egyptian national identity
    document. You must extract the fields without making any changes to the fields and return them
    as they are in Arabic script. If there is something you cannot extract, do not attempt to infer it based
    on other information.
'''
USER_PROMPT = f'''
    You are given two sides of an Egyptian National ID, front and back.
    Extract all the fields, regardless of the side they are found on, out of the ID 
    following this format: {field_structure}.  The key order does not matter. Return all 
    fields as they appear and do not make any changes or updates to any of the fields. Return in json format.
'''

In [ ]:
def preprocess_image(image: Image):
    grey_image = image.convert('L')
    enhancer = ImageEnhance.Contrast(grey_image)
    enhanced_image = enhancer.enhance(1.5)

    return enhanced_image

In [ ]:
def generate_conversation(data):

    image_front = preprocess_image(Image.open(f"./data/synthetic-ids/images/{data['image_front']}"))
    image_back = preprocess_image(Image.open(f"./data/synthetic-ids/images/{data['image_back']}"))
    
    first_name = data['first_name']
    last_name = data['last_name']
    gender = data['gender']
    national_id = data['national_id']
    address = data['address']
    issue_date = data['issue_date']
    expiration_date = data['expiration_date']
    job_title = data['job_title']
    birthdate = data['birthdate']
    religion = data['religion']
    marital_status = data['marital_status']
    address2 = data['address2']

    message = f'''{{"first_name": "{first_name}",
        "last_name": "{last_name}",
        "national_id": "{national_id}",
        "address": "{address}",
        "address2": "{address2}",
        "birthdate": "{birthdate}",
        "issue_date": "{issue_date}",
        "expiration_date": "{expiration_date}",
        "gender": "{gender}",
        "job_title": "{job_title}",
        "religion": "{religion}",
        "marital_status": "{marital_status}"}}'''

    conversation = [
        {
            'role': 'system',
            'content': [
                    {
                        'type': 'text',
                        'text': SYSTEM_PROMPT
                    }
            ]
        },
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text', 'text': USER_PROMPT 
                },
                {
                    'type': 'image', 
                    'image': image_front
                },
                {
                    'type': 'image',
                    'image': image_back
                }
            ]
        },
        {
            'role': 'assistant', 
            'content': [
                {
                    'type': 'text', 
                    'text': message
                }
            ]
        }
    ]
    return {"messages": conversation}

##### Load and split data

In [ ]:
dataset = pd.read_csv("./data/synthetic-ids/IDLabels.csv")

In [ ]:
# TODO: update to load images and fields separately
train, val = train_test_split(dataset, test_size=0.25)

#### Apply chat transformation

In [ ]:
training_data = []
for idx, sample in train.iterrows():
    training_data.append(generate_conversation(sample))

In [ ]:
validation_data = []
for idx, sample in val.iterrows():
    validation_data.append(generate_conversation(sample))

#### Define inference functions

In [ ]:
def infer(model, tokenizer, sample):

    FastVisionModel.for_inference(model)

    preprocessed_front, preprocessed_back = preprocess_image(sample[0]), preprocess_image(sample[1])
    
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_PROMPT}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT}
            ]
        }
    ]

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        [preprocessed_front, preprocessed_back],  
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=256,     
        use_cache=True,
        do_sample=False,       
    )

    input_length = inputs["input_ids"].shape[1] # calculate inpute tokens length to skip in output
    inference = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)

    return inference

In [ ]:
def batch_infer(model, tokenizer, samples):
    FastVisionModel.for_inference(model)
    predictions = []
    
    for sample in tqdm(samples):
        prediction = infer(model, tokenizer, sample)
        predictions.append(prediction)
        
    return predictions


#### Define evaluation metrics

In [ ]:
def eval(model, tokenizer, samples):
    sample_images = [(Image.open(f"./data/synthetic-ids/images/{s['image_front']}"), Image.open(f"./data/synthetic-ids/images/{s['image_back']}")) for i, s in samples.iterrows()]
    predictions = batch_infer(model, tokenizer, sample_images)

    field_correct = {key: 0 for key in field_structure}
    field_avg_wer = {key: 0 for key in field_structure if key != "national_id" and "date" not in key} # no words in national_id or any date field
    field_avg_cer = {key: 0 for key in field_structure}
    total = len(samples)

    assert len(samples) == len(predictions)

    for sample, prediction in tqdm(zip(samples, predictions)):
        for key in field_correct:
            field_correct[key] += 1 if prediction[key] == sample[key] else 0

        for key in field_avg_wer:
            field_avg_wer[key] += wer(prediction[key], sample[key])

        for key in field_avg_cer:
            field_avg_cer[key] += cer(prediction[key], sample[key])
        

    for key in field_correct:
        field_correct[key] /= total
        field_avg_cer[key] /= total
    for key in field_avg_wer:
        field_avg_cer[key] /= total

    return {"correct_match": field_correct, "avg_cer": field_avg_cer, "avg_wer": field_avg_wer}
        

##### Load pretrained model

In [ ]:
# model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
#                                                    load_in_4bit = True,
#                                                    use_gradient_checkpointing=True)


model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",
                                                   load_in_4bit = True,
                                                   use_gradient_checkpointing=True)

##### Set up finetuning model

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)
FastVisionModel.for_training(model)

In [ ]:
args = SFTConfig(
        # training
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        learning_rate = 2e-4, 
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",

        # eval
        per_device_eval_batch_size = 1,
        eval_strategy='steps',
        eval_steps=50,

        # output
        output_dir = "models",
        report_to = 'wandb',
        run_name='ocr-id-detection',

        # logging
        logging_steps = 25,
        save_steps=50,

        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 10000,
        bf16=False,

        push_to_hub=True,
        hub_private_repo=True,
        hub_model_id='zain110506/ocr-id-parser',
        hub_strategy='checkpoint'
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_data,
    eval_dataset=validation_data,
    args=args
)


#### Train the model

In [ ]:
trainer_stats = trainer.train()

#### Save the model and tokenizer

In [ ]:
model.save_pretrained("qwen3_vlm")
tokenizer.save_pretrained("qwen3_vlm")